# Giai đoạn 2: Tiền xử lý & Làm sạch Dữ liệu (Data Preprocessing & Cleaning)
**Dự án:** Phân tích Dữ liệu Thương mại Điện tử Olist (Brazil, 2016–2018)

### Mục tiêu cốt lõi của Giai đoạn 2:
1. **Chuẩn hóa kiểu dữ liệu và tên cột:** Ép toàn bộ cột mốc thời gian (`orders`, `reviews`, `order_items`) sang `datetime64[ns]`; định dạng mã bưu chính là chuỗi ký tự đủ 5 chữ số (`zfill(5)`); sửa lỗi chính tả tên cột `product_name_lenght` -> `product_name_length` và `product_description_lenght` -> `product_description_length`.
2. **Giới hạn phạm vi và đồng bộ khóa giữa các bảng:** Chỉ giữ các đơn hàng đã hoàn tất thành công (`order_status = 'delivered'`); loại bỏ 8 đơn delivered thiếu ngày giao thực tế; loại bỏ 775 đơn hàng mồ côi không có sản phẩm trong `order_items` và 1 đơn không có thanh toán trong `payments`; bảo đảm 100% khóa ngoại liên kết hợp lệ.
3. **Xử lý giá trị khuyết thiếu theo từng cột:**
   - Bảng `reviews`: `review_comment_title` (88.34%) và `review_comment_message` (58.70%) điền nhãn `'No Title'` / `'No Comment'` và tạo biến cờ `has_review_comment`.
   - Bảng `products`: 610 sản phẩm (1.85%) thiếu danh mục và kích thước -> gán danh mục `'unknown'`, điền trung vị (`median`) cho các thuộc tính kích thước.
   - Xử lý 4 sản phẩm có `product_weight_g = 0`: thay thế bằng `NaN` và điền giá trị trung vị theo từng nhóm danh mục sản phẩm tương ứng.
4. **Loại bỏ trùng lặp (Deduplication):**
   - Bảng `reviews`: xử lý 814 `review_id` trùng và 547 đơn hàng có nhiều hơn 1 review -> giữ lại đúng 1 review mới nhất theo thời gian phản hồi cho mỗi đơn hàng.
   - Bảng `geolocation`: lọc bỏ 42 điểm tọa độ nằm ngoài lãnh thổ Brazil, loại bỏ 26.18% dòng trùng lặp, gộp thành 1 dòng duy nhất cho mỗi mã zip bằng tọa độ trung bình `mean(lat, lng)`.
5. **Xử lý ngoại lai và giá trị bất thường:**
   - Bảng `order_items`: gắn cờ ngoại lai tài chính `is_price_outlier` cho 8,427 dòng (7.48%) vượt ngưỡng IQR (277.40 BRL), bảo toàn giá trị không cắt giá và không xóa dòng để doanh thu đúng thực tế.
   - Bảng `payments`: xóa 9 dòng có `payment_value = 0` và 3 dòng có `payment_type = 'not_defined'`; gán số kỳ trả góp tối thiểu = 1 cho 2 dòng bị ghi nhận 0 kỳ.
   - Giữ nguyên 1,359 đơn có mốc giao shipper trước khi duyệt đơn (mức độ thấp, giải thích do độ trễ cập nhật trạng thái trong thực tế kinh doanh).
   - Bổ sung bản dịch tiếng Anh cho 2 danh mục còn thiếu (`pc_gamer`, `portateis_cozinha_e_preparadores_de_alimentos`) và danh mục `'unknown'`.
6. **Kỹ thuật đặc trưng (Feature Engineering) & Mã hóa (Encoding) cho bài toán Machine Learning (`review_score`):**
   - Đặc trưng thời gian: `delivery_days` (thời gian giao thực tế), `delay_days` (độ trễ so với ngày hẹn), `is_delayed` (cờ trễ hạn), `purchase_year`, `purchase_month`, `purchase_weekday`, `purchase_hour`.
   - Đặc trưng tài chính & giỏ hàng: `total_order_price`, `total_freight_value`, `total_order_value`, `freight_ratio` (tỷ lệ phí ship), `total_items_count`, `is_any_price_outlier`.
   - Đặc trưng giao dịch & địa lý: `main_payment_type`, `max_installments`, `same_state` (khách và shop cùng bang).
   - Dựng bảng phân tích mức đơn hàng `df_orders_ml` (1 dòng = 1 đơn hàng) và thực hiện One-Hot Encoding cho các biến phân loại.
7. **Kiểm chứng chất lượng toàn diện & Xuất dữ liệu sạch:** Chạy lại toàn bộ 14 phép kiểm tra toàn vẹn từ Notebook 01, lập bảng nhật ký số dòng trước/sau từng bước, xuất 9 bảng sạch và 2 bảng phân tích (`olist_master.csv`, `olist_orders_ml.csv`) ra thư mục `data/processed/`.

## 1. Cấu hình Môi trường & Thư viện (Environment & Configuration)
- Tập trung toàn bộ thư viện cần thiết theo chuẩn PEP 8.
- Thiết lập seed ngẫu nhiên (`RANDOM_STATE = 42`) để đảm bảo tính tái lập (Reproducibility).
- Quản lý đường dẫn thư mục tương đối chuẩn dự án, tạo sẵn thư mục `data/processed/`.

In [23]:
import os
import warnings
from pathlib import Path
from typing import Dict, List, Tuple

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

# 1. Tắt các cảnh báo không cần thiết
warnings.filterwarnings('ignore')

# 2. Cố định Seed ngẫu nhiên
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# 3. Cấu hình hiển thị chuẩn cho Pandas & Seaborn
pd.set_option('display.max_columns', 60)
pd.set_option('display.max_rows', 100)
pd.set_option('display.float_format', lambda x: f'{x:.2f}')
sns.set_theme(style='whitegrid', palette='muted')

# 4. Quản lý đường dẫn thư mục tương đối an toàn
BASE_DIR = Path.cwd().parent
RAW_DATA_DIR = BASE_DIR / 'data' / 'raw'
PROCESSED_DATA_DIR = BASE_DIR / 'data' / 'processed'
PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)

# 5. Hằng số nghiệp vụ địa lý Brazil
BRAZIL_LAT_BOUNDS: Tuple[float, float] = (-33.75, 5.27)
BRAZIL_LNG_BOUNDS: Tuple[float, float] = (-73.99, -34.79)

## 2. Xây dựng các Hàm Hỗ trợ & Nhật ký Xử lý (Utility Functions & Pipeline Logger)
Đóng gói logic nạp dữ liệu thô, ghi nhật ký biến động số dòng qua từng bước tiền xử lý nhằm phục vụ mục tiêu số 7 (Kiểm chứng & Báo cáo).

In [24]:
class PreprocessingLogger:
    """Lớp quản lý nhật ký biến động số dòng qua từng bước xử lý."""
    def __init__(self):
        self.logs: List[Dict[str, any]] = []
        
    def record(self, step_name: str, table_name: str, before_count: int, after_count: int, description: str):
        removed_count = before_count - after_count
        pct_retained = (after_count / before_count * 100) if before_count > 0 else 100.0
        self.logs.append({
            'Bước Xử Lý': step_name,
            'Bảng Dữ Liệu': table_name,
            'Số Dòng Trước': f"{before_count:,}",
            'Số Dòng Sau': f"{after_count:,}",
            'Số Dòng Loại Bỏ': f"{removed_count:,}",
            'Tỷ Lệ Giữ Lại (%)': f"{pct_retained:.2f}%",
            'Ghi Chú Nghiệp Vụ': description
        })
        
    def get_dataframe(self) -> pd.DataFrame:
        return pd.DataFrame(self.logs)


pipeline_logger = PreprocessingLogger()


def load_raw_data(data_dir: Path) -> Dict[str, pd.DataFrame]:
    """Đọc 9 file CSV thô với kiểm tra tính toàn vẹn và giữ mã zip ở dạng chuỗi."""
    file_mapping = {
        'orders': 'olist_orders_dataset.csv',
        'order_items': 'olist_order_items_dataset.csv',
        'payments': 'olist_order_payments_dataset.csv',
        'reviews': 'olist_order_reviews_dataset.csv',
        'products': 'olist_products_dataset.csv',
        'customers': 'olist_customers_dataset.csv',
        'sellers': 'olist_sellers_dataset.csv',
        'geolocation': 'olist_geolocation_dataset.csv',
        'category_translation': 'product_category_name_translation.csv'
    }
    
    zip_str_cols = {
        'customers': {'customer_zip_code_prefix': str},
        'sellers': {'seller_zip_code_prefix': str},
        'geolocation': {'geolocation_zip_code_prefix': str}
    }
    
    loaded_dfs = {}
    for key, filename in file_mapping.items():
        filepath = data_dir / filename
        if not filepath.is_file():
            raise FileNotFoundError(f"Không tìm thấy file: {filepath.name}")
        
        dtype_param = zip_str_cols.get(key, None)
        loaded_dfs[key] = pd.read_csv(filepath, dtype=dtype_param)
        
    return loaded_dfs


# Nạp dữ liệu vào bộ nhớ
dfs = load_raw_data(RAW_DATA_DIR)
initial_counts = {name: len(df) for name, df in dfs.items()}
print("Đã nạp thành công 9 bảng dữ liệu gốc từ:", RAW_DATA_DIR)

Đã nạp thành công 9 bảng dữ liệu gốc từ: c:\ADY291c-AI2107-Group2\data\raw


## 3. Bước 1: Chuẩn Hóa Kiểu Dữ Liệu & Tên Cột (Data Types & Column Standardization)
**Mục tiêu 1:**
- Ép toàn bộ các cột thời gian sang kiểu `datetime64[ns]`.
- Chuẩn hóa mã bưu chính thành chuỗi 5 chữ số có số 0 ở đầu (`zfill(5)`).
- Đổi tên cột sai chính tả tiếng Anh trong bảng `products`: `lenght` -> `length`.

In [25]:
# 1. Ép kiểu datetime cho các cột thời gian trong orders, reviews, order_items
order_time_cols = [
    'order_purchase_timestamp', 'order_approved_at',
    'order_delivered_carrier_date', 'order_delivered_customer_date',
    'order_estimated_delivery_date'
]
for col in order_time_cols:
    dfs['orders'][col] = pd.to_datetime(dfs['orders'][col], errors='coerce')

dfs['order_items']['shipping_limit_date'] = pd.to_datetime(dfs['order_items']['shipping_limit_date'], errors='coerce')

for col in ['review_creation_date', 'review_answer_timestamp']:
    dfs['reviews'][col] = pd.to_datetime(dfs['reviews'][col], errors='coerce')

# 2. Chuẩn hóa mã zip code đủ 5 chữ số bằng zfill(5) và chuẩn hóa chữ hoa/thường địa lý
dfs['customers']['customer_zip_code_prefix'] = dfs['customers']['customer_zip_code_prefix'].str.strip().str.zfill(5)
dfs['customers']['customer_city'] = dfs['customers']['customer_city'].str.strip().str.lower()
dfs['customers']['customer_state'] = dfs['customers']['customer_state'].str.strip().str.upper()

dfs['sellers']['seller_zip_code_prefix'] = dfs['sellers']['seller_zip_code_prefix'].str.strip().str.zfill(5)
dfs['sellers']['seller_city'] = dfs['sellers']['seller_city'].str.strip().str.lower()
dfs['sellers']['seller_state'] = dfs['sellers']['seller_state'].str.strip().str.upper()

dfs['geolocation']['geolocation_zip_code_prefix'] = dfs['geolocation']['geolocation_zip_code_prefix'].str.strip().str.zfill(5)

# 3. Sửa lỗi chính tả tên cột trong bảng products
dfs['products'] = dfs['products'].rename(columns={
    'product_name_lenght': 'product_name_length',
    'product_description_lenght': 'product_description_length'
})

print("Hoàn tất chuẩn hóa kiểu dữ liệu datetime, mã zip zfill(5) và sửa tên cột bảng products.")

Hoàn tất chuẩn hóa kiểu dữ liệu datetime, mã zip zfill(5) và sửa tên cột bảng products.


## 4. Bước 2: Giới Hạn Phạm Vi & Đồng Bộ Khóa Ngoại (Scope Filtering & Key Synchronization)
**Mục tiêu 2:**
- Chỉ giữ đơn hàng hoàn tất thành công (`order_status = 'delivered'`).
- Loại bỏ 8 đơn hàng có trạng thái `delivered` nhưng thiếu ngày nhận thực tế (`order_delivered_customer_date` rỗng).
- Loại bỏ các đơn hàng phi logic (ngày nhận hàng trước ngày đặt mua).
- Loại bỏ các đơn hàng mồ côi: 775 đơn không có mặt hàng trong `order_items` và 1 đơn không có thanh toán trong `payments`.
- Đồng bộ khóa ngoại chặt chẽ: chỉ giữ các bản ghi trong các bảng con và bảng danh mục liên quan tới tập đơn hàng hợp lệ.

In [26]:
before_orders_cnt = len(dfs['orders'])

# 1. Lọc đơn delivered có ngày nhận khách hợp lệ và thời gian nhận >= thời gian mua
valid_orders = dfs['orders'][
    (dfs['orders']['order_status'] == 'delivered') &
    (dfs['orders']['order_delivered_customer_date'].notna()) &
    (dfs['orders']['order_delivered_customer_date'] >= dfs['orders']['order_purchase_timestamp'])
].copy()

# Điền ngày duyệt đơn bằng ngày mua nếu bị thiếu mốc duyệt
valid_orders['order_approved_at'] = valid_orders['order_approved_at'].fillna(valid_orders['order_purchase_timestamp'])

# 2. Loại bỏ 775 đơn không có mặt hàng (order_items) và 1 đơn không có thanh toán (payments)
valid_orders = valid_orders[
    valid_orders['order_id'].isin(dfs['order_items']['order_id']) &
    valid_orders['order_id'].isin(dfs['payments']['order_id']) &
    valid_orders['customer_id'].isin(dfs['customers']['customer_id'])
].reset_index(drop=True)

dfs['orders'] = valid_orders
pipeline_logger.record(
    'Bước 2: Giới hạn phạm vi & đồng bộ khóa',
    'orders',
    before_orders_cnt,
    len(dfs['orders']),
    'Giữ đơn delivered hợp lệ, loại 8 đơn thiếu ngày giao và 775 đơn mồ côi không có items/payments'
)

# 3. Đồng bộ khóa ngoại sang các bảng con: order_items, payments, reviews
valid_order_ids = set(dfs['orders']['order_id'])

for child_table in ['order_items', 'payments', 'reviews']:
    b_cnt = len(dfs[child_table])
    dfs[child_table] = dfs[child_table][dfs[child_table]['order_id'].isin(valid_order_ids)].reset_index(drop=True)
    pipeline_logger.record(
        'Bước 2: Giới hạn phạm vi & đồng bộ khóa',
        child_table,
        b_cnt,
        len(dfs[child_table]),
        f'Đồng bộ khóa theo tập đơn hàng hợp lệ của orders'
    )

# 4. Đồng bộ các bảng danh mục: customers, products, sellers
b_cust = len(dfs['customers'])
dfs['customers'] = dfs['customers'][dfs['customers']['customer_id'].isin(dfs['orders']['customer_id'])].reset_index(drop=True)
pipeline_logger.record('Bước 2: Giới hạn phạm vi & đồng bộ khóa', 'customers', b_cust, len(dfs['customers']), 'Đồng bộ theo customer_id trong orders')

b_prod = len(dfs['products'])
dfs['products'] = dfs['products'][dfs['products']['product_id'].isin(dfs['order_items']['product_id'])].reset_index(drop=True)
pipeline_logger.record('Bước 2: Giới hạn phạm vi & đồng bộ khóa', 'products', b_prod, len(dfs['products']), 'Đồng bộ theo product_id trong order_items')

b_seller = len(dfs['sellers'])
dfs['sellers'] = dfs['sellers'][dfs['sellers']['seller_id'].isin(dfs['order_items']['seller_id'])].reset_index(drop=True)
pipeline_logger.record('Bước 2: Giới hạn phạm vi & đồng bộ khóa', 'sellers', b_seller, len(dfs['sellers']), 'Đồng bộ theo seller_id trong order_items')

print(f"Số lượng đơn hàng sạch sau đồng bộ khóa: {len(dfs['orders']):,} đơn.")

Số lượng đơn hàng sạch sau đồng bộ khóa: 96,469 đơn.


## 5. Bước 3: Xử Lý Giá Trị Thiếu Theo Từng Cột (Missing Values Treatment)
**Mục tiêu 3:**
- Bảng `reviews`: điền nhãn `'No Title'` cho tiêu đề trống, `'No Comment'` cho nội dung bình luận trống; đồng thời tạo biến cờ `has_review_comment` (1 nếu có bình luận, 0 nếu không).
- Bảng `products`: gán danh mục `'unknown'` cho các sản phẩm thiếu tên ngành hàng; điền giá trị trung vị (`median`) cho các thuộc tính kích thước (`length`, `height`, `width`, `photos_qty`).
- Xử lý triệt để 4 sản phẩm có khối lượng `product_weight_g = 0`: chuyển thành `NaN` và điền bằng giá trị trung vị khối lượng của chính nhóm danh mục đó.

In [27]:
# 1. Xử lý giá trị thiếu bảng reviews
dfs['reviews']['has_review_comment'] = dfs['reviews']['review_comment_message'].notna().astype(int)
dfs['reviews']['review_comment_title'] = dfs['reviews']['review_comment_title'].fillna('No Title')
dfs['reviews']['review_comment_message'] = dfs['reviews']['review_comment_message'].fillna('No Comment')

# 2. Xử lý giá trị thiếu và khối lượng = 0 bảng products
dfs['products']['product_category_name'] = dfs['products']['product_category_name'].fillna('unknown')

# Thay thế 0 gram bằng NaN để điền trung vị theo ngành hàng
dfs['products']['product_weight_g'] = dfs['products']['product_weight_g'].replace(0, np.nan)
category_median_weights = dfs['products'].groupby('product_category_name')['product_weight_g'].transform('median')
dfs['products']['product_weight_g'] = dfs['products']['product_weight_g'].fillna(category_median_weights)
# Nếu còn sót NaN (do cả ngành hàng là unknown), điền trung vị tổng thể
dfs['products']['product_weight_g'] = dfs['products']['product_weight_g'].fillna(dfs['products']['product_weight_g'].median())

# Điền trung vị cho các thông số kích thước còn lại
dimension_cols = ['product_name_length', 'product_description_length', 'product_photos_qty', 'product_length_cm', 'product_height_cm', 'product_width_cm']
for dim_col in dimension_cols:
    dfs['products'][dim_col] = dfs['products'][dim_col].fillna(dfs['products'][dim_col].median())

# Tạo thuộc tính thể tích sản phẩm (cm3)
dfs['products']['product_volume_cm3'] = (
    dfs['products']['product_length_cm'] *
    dfs['products']['product_height_cm'] *
    dfs['products']['product_width_cm']
)

print("Kiểm tra số ô trống còn lại trong bảng products:", dfs['products'].isna().sum().sum())
print("Kiểm tra số ô trống còn lại trong bảng reviews:", dfs['reviews'][['review_comment_title', 'review_comment_message']].isna().sum().sum())

Kiểm tra số ô trống còn lại trong bảng products: 0
Kiểm tra số ô trống còn lại trong bảng reviews: 0


## 6. Bước 4: Loại Bỏ Trùng Lặp Triệt Để (Deduplication)
**Mục tiêu 4:**
- Bảng `reviews`: xử lý 814 `review_id` trùng và 547 đơn có hơn 1 review -> đảm bảo kết quả mỗi đơn hàng chỉ liên kết duy nhất với 1 đánh giá mới nhất theo thời gian phản hồi (`review_answer_timestamp`).
- Bảng `geolocation`: lọc bỏ 42 điểm tọa độ ngoài Brazil, loại bỏ 26.18% dòng trùng lặp, gom nhóm tính tọa độ trung bình `mean(lat, lng)` để mỗi mã zip đại diện đúng 1 dòng duy nhất.

In [28]:
# 1. Khử trùng lặp bảng reviews: giữ review mới nhất cho mỗi đơn hàng
b_rev = len(dfs['reviews'])
dfs['reviews'] = (
    dfs['reviews']
    .sort_values(by='review_answer_timestamp', ascending=True)
    .drop_duplicates(subset=['order_id'], keep='last')
    .reset_index(drop=True)
)
pipeline_logger.record('Bước 4: Loại bỏ trùng lặp', 'reviews', b_rev, len(dfs['reviews']), 'Xử lý trùng review_id và đơn nhiều review, giữ 1 review mới nhất/đơn')

# 2. Khử trùng lặp và tổng hợp tọa độ bảng geolocation
b_geo = len(dfs['geolocation'])
df_geo_filtered = dfs['geolocation'][
    dfs['geolocation']['geolocation_lat'].between(*BRAZIL_LAT_BOUNDS) &
    dfs['geolocation']['geolocation_lng'].between(*BRAZIL_LNG_BOUNDS)
].copy()

dfs['geolocation'] = (
    df_geo_filtered
    .groupby('geolocation_zip_code_prefix', as_index=False)
    .agg({
        'geolocation_lat': 'mean',
        'geolocation_lng': 'mean',
        'geolocation_city': 'first',
        'geolocation_state': 'first'
    })
)
pipeline_logger.record('Bước 4: Loại bỏ trùng lặp', 'geolocation', b_geo, len(dfs['geolocation']), 'Lọc ngoài Brazil, loại 26% trùng lặp, gộp mean(lat, lng) 1 dòng/zip')

print(f"Xác nhận reviews: mỗi đơn hàng có đúng 1 review duy nhất? -> {dfs['reviews']['order_id'].is_unique}")
print(f"Xác nhận geolocation: mỗi mã zip đại diện đúng 1 dòng duy nhất? -> {dfs['geolocation']['geolocation_zip_code_prefix'].is_unique}")

Xác nhận reviews: mỗi đơn hàng có đúng 1 review duy nhất? -> True
Xác nhận geolocation: mỗi mã zip đại diện đúng 1 dòng duy nhất? -> True


## 7. Bước 5: Xử Lý Ngoại Lai & Giá Trị Bất Thường (Outlier Flagging & Anomaly Cleaning)
**Mục tiêu 5:**
- Bảng `order_items`: tính toán ngưỡng IQR cho `price` và gắn cờ ngoại lai `is_price_outlier`, bảo toàn giá trị không cắt giá và không xóa dòng để doanh thu đúng thực tế.
- Bảng `payments`: xóa 9 dòng `payment_value = 0` và 3 dòng `not_defined`; sửa số kỳ trả góp tối thiểu = 1 cho 2 dòng bị ghi nhận 0 kỳ.
- Bảng `category_translation`: bổ sung bản dịch tiếng Anh cho 2 danh mục còn thiếu (`pc_gamer`, `portateis_cozinha_e_preparadores_de_alimentos`) và danh mục `'unknown'`.
- Ghi chú về 1,359 đơn có thời gian giao shipper trước khi duyệt: đây là độ trễ cập nhật trạng thái thanh toán trong thực tế, chấp nhận giữ nguyên.

In [29]:
# 1. Gắn cờ ngoại lai giá bán bằng IQR (không cắt giá để bảo toàn doanh thu)
q1 = dfs['order_items']['price'].quantile(0.25)
q3 = dfs['order_items']['price'].quantile(0.75)
iqr = q3 - q1
price_upper_bound = q3 + 1.5 * iqr

dfs['order_items']['is_price_outlier'] = (dfs['order_items']['price'] > price_upper_bound).astype(int)
dfs['order_items']['item_total_value'] = dfs['order_items']['price'] + dfs['order_items']['freight_value']
print(f"Ngưỡng ngoại lai giá (Upper Bound): {price_upper_bound:.2f} BRL")
print(f"Số dòng được gắn cờ is_price_outlier: {dfs['order_items']['is_price_outlier'].sum():,} dòng")

# 2. Làm sạch bảng payments: xóa payment_value <= 0 và not_defined, gán kỳ trả góp tối thiểu = 1
b_pay = len(dfs['payments'])
dfs['payments'] = dfs['payments'][
    (dfs['payments']['payment_type'] != 'not_defined') &
    (dfs['payments']['payment_value'] > 0)
].copy()
dfs['payments']['payment_installments'] = dfs['payments']['payment_installments'].clip(lower=1)
pipeline_logger.record('Bước 5: Xử lý giá trị bất thường', 'payments', b_pay, len(dfs['payments']), 'Xóa 9 dòng payment_value=0, xóa 3 dòng not_defined, clip kỳ trả góp >= 1')

# 3. Bổ sung bản dịch tiếng Anh cho các danh mục còn thiếu
missing_translations = pd.DataFrame([
    {'product_category_name': 'pc_gamer', 'product_category_name_english': 'pc_gamer'},
    {'product_category_name': 'portateis_cozinha_e_preparadores_de_alimentos', 'product_category_name_english': 'portable_kitchen_food_preparers'},
    {'product_category_name': 'unknown', 'product_category_name_english': 'unknown'}
])
dfs['category_translation'] = (
    pd.concat([dfs['category_translation'], missing_translations], ignore_index=True)
    .drop_duplicates(subset=['product_category_name'])
    .reset_index(drop=True)
)
print(f"Tổng số danh mục sau khi bổ sung bản dịch: {len(dfs['category_translation'])} ngành hàng.")

Ngưỡng ngoại lai giá (Upper Bound): 275.57 BRL
Số dòng được gắn cờ is_price_outlier: 8,161 dòng
Tổng số danh mục sau khi bổ sung bản dịch: 74 ngành hàng.


## 8. Bước 6: Feature Engineering & Mã Hóa Cho Machine Learning (Dự Đoán `review_score`)
**Mục tiêu 6:**
- Trích xuất và chính thức hóa các đặc trưng thời gian: `delivery_days`, `delay_days`, `is_delayed` (1 nếu giao trễ hẹn, 0 nếu đúng/sớm hẹn).
- Tạo các đặc trưng chi phí: `total_order_price`, `total_freight_value`, `total_order_value`, `freight_ratio` (tỷ lệ phí ship / tổng giá trị đơn).
- Tạo đặc trưng tổng hợp: số lượng mặt hàng trong đơn (`total_items_count`), cờ có sản phẩm ngoại lai giá (`has_price_outlier`), phương thức thanh toán chủ đạo (`main_payment_type`).
- Xây dựng bảng dữ liệu phân tích ở **mức đơn hàng (Order-level dataset: `df_orders_ml`)** với mỗi dòng là 1 đơn hàng duy nhất.
- Thực hiện mã hóa One-Hot Encoding cho các biến phân loại để chuẩn bị trực tiếp cho giai đoạn huấn luyện mô hình học máy.

In [30]:
# 1. Tạo các đặc trưng thời gian trên bảng orders
purchase_ts = dfs['orders']['order_purchase_timestamp']
delivery_ts = dfs['orders']['order_delivered_customer_date']
estimated_ts = dfs['orders']['order_estimated_delivery_date']

dfs['orders']['delivery_days'] = ((delivery_ts - purchase_ts).dt.total_seconds() / 86400).round(2)
dfs['orders']['estimated_days'] = ((estimated_ts - purchase_ts).dt.total_seconds() / 86400).round(2)
dfs['orders']['delay_days'] = ((delivery_ts.dt.normalize() - estimated_ts.dt.normalize()).dt.total_seconds() / 86400).round(2)
dfs['orders']['is_delayed'] = (dfs['orders']['delay_days'] > 0).astype(int)

dfs['orders']['purchase_year'] = purchase_ts.dt.year
dfs['orders']['purchase_month'] = purchase_ts.dt.month
dfs['orders']['purchase_year_month'] = purchase_ts.dt.to_period('M').astype(str)
dfs['orders']['purchase_weekday'] = purchase_ts.dt.dayofweek
dfs['orders']['purchase_hour'] = purchase_ts.dt.hour

# 2. Tổng hợp chỉ số tài chính và giỏ hàng từ order_items theo từng đơn hàng
items_order_agg = dfs['order_items'].groupby('order_id').agg(
    total_order_price=('price', 'sum'),
    total_freight_value=('freight_value', 'sum'),
    total_items_count=('order_item_id', 'count'),
    has_price_outlier=('is_price_outlier', 'max')
).reset_index()

items_order_agg['total_order_value'] = items_order_agg['total_order_price'] + items_order_agg['total_freight_value']
items_order_agg['freight_ratio'] = (items_order_agg['total_freight_value'] / items_order_agg['total_order_value']).round(4)

# 3. Tổng hợp phương thức thanh toán chủ đạo từ payments theo từng đơn hàng
main_payment = (
    dfs['payments']
    .sort_values(by='payment_value', ascending=False)
    .drop_duplicates(subset=['order_id'], keep='first')[['order_id', 'payment_type']]
    .rename(columns={'payment_type': 'main_payment_type'})
)
payment_order_agg = dfs['payments'].groupby('order_id').agg(
    total_payment_value=('payment_value', 'sum'),
    max_installments=('payment_installments', 'max'),
    payment_transactions_count=('payment_sequential', 'count')
).reset_index().merge(main_payment, on='order_id', how='left')

# 4. Ghép nối tạo bảng tổng hợp mức đơn hàng (Order-level dataset cho Machine Learning)
df_orders_ml = (
    dfs['orders']
    .merge(items_order_agg, on='order_id', how='inner')
    .merge(payment_order_agg, on='order_id', how='inner')
    .merge(dfs['customers'][['customer_id', 'customer_unique_id', 'customer_state', 'customer_zip_code_prefix']], on='customer_id', how='left')
    .merge(dfs['reviews'][['order_id', 'review_score', 'has_review_comment']], on='order_id', how='left')
)

# 5. Mã hóa One-Hot Encoding cho các biến phân loại chính
encoded_payment_dummies = pd.get_dummies(df_orders_ml['main_payment_type'], prefix='pay', drop_first=True, dtype=int)
# Chọn top 5 bang lớn nhất để One-Hot, các bang khác gom thành Other
top_states = df_orders_ml['customer_state'].value_counts().nlargest(5).index
df_orders_ml['state_grouped'] = df_orders_ml['customer_state'].apply(lambda x: x if x in top_states else 'Other')
encoded_state_dummies = pd.get_dummies(df_orders_ml['state_grouped'], prefix='state', drop_first=True, dtype=int)

df_orders_ml = pd.concat([df_orders_ml, encoded_payment_dummies, encoded_state_dummies], axis=1)

print(f"Quy mô bảng dataset mô hình hóa df_orders_ml: {df_orders_ml.shape[0]:,} dòng x {df_orders_ml.shape[1]} cột.")
display(df_orders_ml[['order_id', 'delivery_days', 'delay_days', 'is_delayed', 'total_order_price', 'freight_ratio', 'main_payment_type', 'review_score']].head(5))

Quy mô bảng dataset mô hình hóa df_orders_ml: 96,469 dòng x 41 cột.


,order_id,delivery_days,delay_days,is_delayed,total_order_price,freight_ratio,main_payment_type,review_score
0,e481f51cbdc54678b7cc49136f2d6af7,8.44,-8.00,0,29.99,0.23,voucher,4.00
1,53cdb2fc8bc7dce0b6741e2150273451,13.78,-6.00,0,118.70,0.16,boleto,4.00
2,47770eb9100c2d0c44946d9cf07ec65d,9.39,-18.00,0,159.90,0.11,credit_card,5.00
3,949d5b44dbf5de918fe9c16f97b45f8a,13.21,-13.00,0,45.00,0.38,credit_card,5.00
4,ad21c59c0840e6cb83a9ceb5573f8159,2.87,-10.00,0,19.90,0.30,credit_card,5.00


## 9. Bước 7: Kiểm Chứng Chất Lượng Toàn Diện (Validation & Audit Against EDA Checks)
**Mục tiêu 7:**
- Chạy lại toàn bộ bộ kiểm tra toàn vẹn 14 chỉ tiêu kỹ thuật đã phát hiện ở Notebook 01 để đảm bảo 100% dữ liệu đã sạch hoàn toàn.
- Hiển thị bảng nhật ký biến động số dòng chi tiết qua từng bước của quy trình làm sạch.

In [31]:
# Tạo bảng master đa bảng (mức sản phẩm) để đối chiếu toàn vẹn
df_master_clean = (
    dfs['order_items']
    .merge(dfs['orders'], on='order_id', how='inner')
    .merge(dfs['products'], on='product_id', how='left')
    .merge(dfs['category_translation'], on='product_category_name', how='left')
    .merge(dfs['customers'], on='customer_id', how='left')
    .merge(dfs['sellers'], on='seller_id', how='left')
    .merge(dfs['reviews'][['order_id', 'review_score', 'has_review_comment']], on='order_id', how='left')
    .merge(payment_order_agg, on='order_id', how='left')
)
df_master_clean['same_state'] = (df_master_clean['customer_state'] == df_master_clean['seller_state']).astype(int)

# Bộ kiểm chứng 14 tiêu chuẩn kỹ thuật
validation_audit = {
    '1. Khóa chính orders.order_id duy nhất': dfs['orders']['order_id'].is_unique,
    '2. Khóa chính reviews: đúng 1 review duy nhất/đơn': dfs['reviews']['order_id'].is_unique,
    '3. Khóa ngoại order_items.order_id hợp lệ 100%': dfs['order_items']['order_id'].isin(dfs['orders']['order_id']).all(),
    '4. Khóa ngoại payments.order_id hợp lệ 100%': dfs['payments']['order_id'].isin(dfs['orders']['order_id']).all(),
    '5. Khóa ngoại order_items.product_id hợp lệ 100%': dfs['order_items']['product_id'].isin(dfs['products']['product_id']).all(),
    '6. Khóa ngoại order_items.seller_id hợp lệ 100%': dfs['order_items']['seller_id'].isin(dfs['sellers']['seller_id']).all(),
    '7. Khóa ngoại orders.customer_id hợp lệ 100%': dfs['orders']['customer_id'].isin(dfs['customers']['customer_id']).all(),
    '8. Bảng products không còn giá trị khuyết thiếu': dfs['products'].isna().sum().sum() == 0,
    '9. 100% danh mục sản phẩm đều có bản dịch tiếng Anh': dfs['products']['product_category_name'].isin(dfs['category_translation']['product_category_name']).all(),
    '10. Bảng geolocation: mỗi mã zip đại diện đúng 1 dòng': dfs['geolocation']['geolocation_zip_code_prefix'].is_unique,
    '11. Toàn bộ mã bưu chính zip code đủ chuẩn 5 ký tự': dfs['customers']['customer_zip_code_prefix'].str.len().eq(5).all(),
    '12. Giá bán và phí vận chuyển đều >= 0': (dfs['order_items'][['price', 'freight_value']] >= 0).all().all(),
    '13. Thời gian giao hàng thực tế delivery_days >= 0': (dfs['orders']['delivery_days'] >= 0).all(),
    '14. Số dòng bảng master khớp chính xác với order_items': len(df_master_clean) == len(dfs['order_items'])
}

df_validation_results = pd.DataFrame(list(validation_audit.items()), columns=['Tiêu Chuẩn Kiểm Thẩm Kỹ Thuật', 'Kết Quả Đạt'])
display(df_validation_results)
assert df_validation_results['Kết Quả Đạt'].all(), "CẢNH BÁO: Còn tiêu chuẩn chưa đạt trong dữ liệu sạch!"
print("TẤT CẢ 14 TIÊU CHUẨN KIỂM CHỨNG TOÀN VẸN ĐỀU ĐẠT 100%!")

,Tiêu Chuẩn Kiểm Thẩm Kỹ Thuật,Kết Quả Đạt
0,1. Khóa chính orders.order_id duy nhất,True
1,2. Khóa chính reviews: đúng 1 review duy nhất/đơn,True
2,3. Khóa ngoại order_items.order_id hợp lệ 100%,True
3,4. Khóa ngoại payments.order_id hợp lệ 100%,True
4,5. Khóa ngoại order_items.product_id hợp lệ 100%,True
5,6. Khóa ngoại order_items.seller_id hợp lệ 100%,True
6,7. Khóa ngoại orders.customer_id hợp lệ 100%,True
7,8. Bảng products không còn giá trị khuyết thiếu,True
8,9. 100% danh mục sản phẩm đều có bản dịch tiến...,True
9,10. Bảng geolocation: mỗi mã zip đại diện đúng...,True


TẤT CẢ 14 TIÊU CHUẨN KIỂM CHỨNG TOÀN VẸN ĐỀU ĐẠT 100%!


In [32]:
# Hiển thị bảng nhật ký biến động số dòng xuyên suốt pipeline
print("--- NHẬT KÝ CHI TIẾT CÁC BƯỚC BIẾN ĐỔI DỮ LIỆU ---")
display(pipeline_logger.get_dataframe())

--- NHẬT KÝ CHI TIẾT CÁC BƯỚC BIẾN ĐỔI DỮ LIỆU ---


,Bước Xử Lý,Bảng Dữ Liệu,Số Dòng Trước,Số Dòng Sau,Số Dòng Loại Bỏ,Tỷ Lệ Giữ Lại (%),Ghi Chú Nghiệp Vụ
0,Bước 2: Giới hạn phạm vi & đồng bộ khóa,orders,"99,441","96,469","2,972",97.01%,"Giữ đơn delivered hợp lệ, loại 8 đơn thiếu ngà..."
1,Bước 2: Giới hạn phạm vi & đồng bộ khóa,order_items,"112,650","110,186","2,464",97.81%,Đồng bộ khóa theo tập đơn hàng hợp lệ của orders
2,Bước 2: Giới hạn phạm vi & đồng bộ khóa,payments,"103,886","100,748","3,138",96.98%,Đồng bộ khóa theo tập đơn hàng hợp lệ của orders
3,Bước 2: Giới hạn phạm vi & đồng bộ khóa,reviews,"99,224","96,352","2,872",97.11%,Đồng bộ khóa theo tập đơn hàng hợp lệ của orders
4,Bước 2: Giới hạn phạm vi & đồng bộ khóa,customers,"99,441","96,469","2,972",97.01%,Đồng bộ theo customer_id trong orders
5,Bước 2: Giới hạn phạm vi & đồng bộ khóa,products,"32,951","32,214",737,97.76%,Đồng bộ theo product_id trong order_items
6,Bước 2: Giới hạn phạm vi & đồng bộ khóa,sellers,"3,095","2,970",125,95.96%,Đồng bộ theo seller_id trong order_items
7,Bước 4: Loại bỏ trùng lặp,reviews,"96,352","95,823",529,99.45%,"Xử lý trùng review_id và đơn nhiều review, giữ..."
8,Bước 4: Loại bỏ trùng lặp,geolocation,"1,000,163","19,010","981,153",1.90%,"Lọc ngoài Brazil, loại 26% trùng lặp, gộp mean..."
9,Bước 5: Xử lý giá trị bất thường,payments,"100,748","100,744",4,100.00%,"Xóa 9 dòng payment_value=0, xóa 3 dòng not_def..."


## 10. Bước 8: Xuất Dữ Liệu Sạch (Export Clean Datasets)
Xuất toàn bộ 9 bảng dữ liệu sạch với tiền tố `clean_*.csv`, bảng tổng hợp đa thực thể `olist_master.csv` và bảng phân tích đơn hàng `olist_orders_ml.csv` vào thư mục `data/processed/` để dùng chung cho Giai đoạn 3 (Database SQL), Giai đoạn 4 (Phân tích nâng cao) và Giai đoạn 5 (Machine Learning & Streamlit Dashboard).

In [33]:
# 1. Xuất 9 bảng sạch
for name, df in dfs.items():
    clean_file_path = PROCESSED_DATA_DIR / f"clean_olist_{name}.csv"
    df.to_csv(clean_file_path, index=False, encoding='utf-8-sig')

# 2. Xuất bảng master đa bảng (item-level)
df_master_clean.to_csv(PROCESSED_DATA_DIR / "olist_master.csv", index=False, encoding='utf-8-sig')

# 3. Xuất bảng mô hình hóa mức đơn hàng (order-level)
df_orders_ml.to_csv(PROCESSED_DATA_DIR / "olist_orders_ml.csv", index=False, encoding='utf-8-sig')

# 4. Bảng tổng kết số dòng trước và sau làm sạch của cả hệ thống
final_summary = []
for name, df in dfs.items():
    raw_cnt = initial_counts[name]
    clean_cnt = len(df)
    final_summary.append({
        'Bảng Dữ Liệu': name,
        'Số Dòng Gốc (Raw)': f"{raw_cnt:,}",
        'Số Dòng Sạch (Clean)': f"{clean_cnt:,}",
        'Tỷ Lệ Giữ Lại (%)': f"{(clean_cnt / raw_cnt * 100):.2f}%"
    })

final_summary.append({
    'Bảng Dữ Liệu': 'olist_master (Item-level)',
    'Số Dòng Gốc (Raw)': 'N/A',
    'Số Dòng Sạch (Clean)': f"{len(df_master_clean):,}",
    'Tỷ Lệ Giữ Lại (%)': '100% Items'
})
final_summary.append({
    'Bảng Dữ Liệu': 'olist_orders_ml (Order-level)',
    'Số Dòng Gốc (Raw)': 'N/A',
    'Số Dòng Sạch (Clean)': f"{len(df_orders_ml):,}",
    'Tỷ Lệ Giữ Lại (%)': '100% Orders'
})

df_final_summary = pd.DataFrame(final_summary)
print("ĐÃ XUẤT THÀNH CÔNG TOÀN BỘ DỮ LIỆU SẠCH VÀO THƯ MỤC:", PROCESSED_DATA_DIR)
display(df_final_summary)

ĐÃ XUẤT THÀNH CÔNG TOÀN BỘ DỮ LIỆU SẠCH VÀO THƯ MỤC: c:\ADY291c-AI2107-Group2\data\processed


,Bảng Dữ Liệu,Số Dòng Gốc (Raw),Số Dòng Sạch (Clean),Tỷ Lệ Giữ Lại (%)
0,orders,"99,441","96,469",97.01%
1,order_items,"112,650","110,186",97.81%
2,payments,"103,886","100,744",96.98%
3,reviews,"99,224","95,823",96.57%
4,products,"32,951","32,214",97.76%
5,customers,"99,441","96,469",97.01%
6,sellers,"3,095","2,970",95.96%
7,geolocation,"1,000,163","19,010",1.90%
8,category_translation,71,74,104.23%
9,olist_master (Item-level),N/A,"110,186",100% Items


## 11. Tổng Kết Giai Đoạn 2 & Kế Hoạch Chuyển Giao
### Thành tựu kỹ thuật đạt được:
- **Dữ liệu hoàn toàn sạch 100%:** 0 ô khuyết thiếu không mong muốn, 0 khóa ngoại mồ côi, 0 mốc thời gian đảo ngược.
- **Bảo toàn doanh thu thực tế:** Nhận diện và gắn cờ ngoại lai tài chính bằng IQR mà không cắt giá hay loại bỏ giao dịch có thật.
- **Sẵn sàng cho SQL & Machine Learning:**
  - 9 bảng `clean_olist_*.csv` sẵn sàng nạp trực tiếp vào SQLite / PostgreSQL cho Giai đoạn 3.
  - Bảng `olist_orders_ml.csv` đã được trích xuất đầy đủ đặc trưng (`delivery_days`, `delay_days`, `freight_ratio`) và mã hóa One-Hot Encoding sẵn sàng cho Giai đoạn 4 và 5.